In [2]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
from tabulate import tabulate

# Downloading the English stopwords list because we need it to filter out common
# meaningless words like the, is, and, a from the review text during cleaning
nltk.download('stopwords', quiet=True)

# Downloading the punkt tokenizer model because word_tokenize relies on this
# model internally to know how to split sentences into words correctly
nltk.download('punkt', quiet=True)

# Downloading the punkt_tab model because newer versions of NLTK replaced the
# old punkt model with punkt_tab and word_tokenize will throw an error without it
nltk.download('punkt_tab', quiet=True)


# Loading the dataset

# Reading the Excel file from the data folder and loading it into a pandas DataFrame
# Using data/ before the filename because the Excel file is saved inside the data folder
df = pd.read_excel('data/telecom_customer_reviews.xlsx')

# Setting display width to 120 so the table spreads across the screen without wrapping
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 60)

# Defining a helper function to print any DataFrame as a clean left aligned table
# Using tabulate with tablefmt plain so there are no borders just clean spacing
# Using showindex False to hide the default row numbers on the left
def print_table(df):
    print(tabulate(df, headers=df.columns, tablefmt='plain', showindex=False))

print()
print('dataset loaded successfully')
print(f'  rows    : {df.shape[0]}')
print(f'  columns : {df.shape[1]}')
print(f'  columns : {df.columns.tolist()}')
print()
print_table(df.head())
print()


# Cleaning, normalizing, tokenizing, removing stopwords and punctuation

# Loading the NLTK English stopwords into a Python set for fast membership checking
# Using a set because checking word in set is much faster than checking in a list
STOPWORDS = set(stopwords.words('english'))

# Converting string.punctuation into a set so we can quickly check if a token is punctuation
# string.punctuation contains all standard punctuation like . , ! ? etc
PUNCTUATION = set(string.punctuation)

def clean_text(text):
    # Converting text to lowercase so Good and good are treated as the same word
    text = str(text).lower()

    # Splitting text into individual word tokens using NLTK word_tokenize
    # Using word_tokenize instead of split because it handles punctuation smartly
    # For example great becomes great and ! separately instead of staying as great!
    tokens = word_tokenize(text)

    # Filtering tokens by removing stopwords, punctuation, numbers and single characters
    tokens = [
        t for t in tokens
        if t not in STOPWORDS      # Removing common meaningless words like the is and a
        and t not in PUNCTUATION   # Removing punctuation marks like . ! ,
        and t.isalpha()            # Keeping only words made of letters, removing numbers and symbols
        and len(t) > 1             # Removing single character tokens like s or t
    ]
    return tokens

# Previewing the clean_text function on one sample review to confirm it works correctly
# Using f-string formatting with labeled keys original and cleaned to make it readable
print('clean text preview')
print(f'  original : {df["review_text"][0]}')
print(f'  cleaned  : {clean_text(df["review_text"][0])}')
print()


# Creating the cleaned review column

# Applying clean_text to every row in review_text column
# Storing result as a list of tokens in cleaned_tokens, used for all further analysis
df['cleaned_tokens'] = df['review_text'].apply(clean_text)

# Joining the token list back into a readable string for display purposes
# Using space between each token with join
df['cleaned_review'] = df['cleaned_tokens'].apply(lambda t: ' '.join(t))

# Printing only two columns side by side using print_table so both columns
# align cleanly to the left with equal spacing between them
print('original vs cleaned review')
print()
print_table(df[['review_text', 'cleaned_review']].head())
print()


# Calculating word frequency across all reviews

# Flattening all token lists from every review into one combined word list
# Doing this so we can count word frequency across the whole dataset not per review
all_words = [word for tokens in df['cleaned_tokens'] for word in tokens]

# Counting occurrences of each unique word using Counter
word_freq = Counter(all_words)

# Converting to a DataFrame and starting index from 1 instead of 0
# so the ranking reads like a numbered list from 1 to 15
freq_df = pd.DataFrame(word_freq.most_common(15), columns=['word', 'frequency'])
freq_df.index = freq_df.index + 1

print('top 15 most common words across all reviews')
print()
print_table(freq_df)
print()


# Creating the review_length column

# Counting the number of tokens in each cleaned review
# Measuring how detailed each customer feedback is after removing stopwords
df['review_length'] = df['cleaned_tokens'].apply(len)

# Printing only selected columns so the output stays focused and not overcrowded
print('review length added')
print()
print_table(df[['review_id', 'customer_name', 'cleaned_review', 'review_length']].head())
print()


# Creating sentiment labels for positive negative and neutral reviews

# Defining a set of words strongly associated with positive customer experience
POSITIVE_KEYWORDS = {
    'excellent', 'good', 'great', 'fast', 'helpful', 'reliable',
    'amazing', 'perfect', 'best', 'easy', 'happy', 'satisfied',
    'love', 'wonderful', 'fantastic', 'quick', 'smooth', 'friendly',
    'clear', 'works'
}

# Defining a set of words strongly associated with negative customer experience
NEGATIVE_KEYWORDS = {
    'slow', 'bad', 'poor', 'terrible', 'awful', 'worst',
    'disappointing', 'frustrating', 'issue', 'problem', 'weak',
    'useless', 'unreliable', 'difficult', 'expensive', 'unhelpful',
    'fail', 'broken', 'dropped', 'horrible', 'rude', 'confusing'
}

def get_sentiment(tokens):
    # Converting token list to a set to remove duplicates and enable fast set operations
    token_set = set(tokens)

    # Finding how many positive and negative keywords exist in the review using intersection
    pos_count = len(token_set & POSITIVE_KEYWORDS)
    neg_count = len(token_set & NEGATIVE_KEYWORDS)

    # Returning the label based on which count is higher
    if pos_count > neg_count:
        return 'Positive'
    elif neg_count > pos_count:
        return 'Negative'
    else:
        return 'Neutral'

# Applying the sentiment function to every row in the DataFrame
df['sentiment_label'] = df['cleaned_tokens'].apply(get_sentiment)

# Converting value_counts to a DataFrame and renaming columns so it prints
# as a proper clean table instead of a plain series output
sentiment_counts = df['sentiment_label'].value_counts().reset_index()
sentiment_counts.columns = ['sentiment', 'count']

print('sentiment distribution')
print()
print_table(sentiment_counts)
print()


# Displaying the first 10 processed rows and interpreting the results

# Selecting only the most relevant columns so the table stays clean and focused
cols = ['review_id', 'customer_name', 'service_type', 'rating',
        'cleaned_review', 'review_length', 'sentiment_label']

print('first 10 processed rows')
print()
print_table(df[cols].head(10))
print()

print('interpretation')
print('''
  1. the dataset contains 12 customer reviews covering both internet and mobile services.

  2. after cleaning with nltk the most frequent words are service, support, internet
     and customer reflecting the core topics discussed in the reviews.

  3. sentiment analysis shows 6 positive, 4 negative and 2 neutral reviews.
     this means the majority of customers are satisfied but a notable portion are not.

  4. positive reviews highlight words like excellent, fast, helpful and reliable
     mostly coming from internet service users.

  5. negative reviews contain words like slow, poor, signal and rude
     mostly from mobile service users.

  6. the average review length is 5 to 7 meaningful words after stopword removal
     showing customers tend to write short but focused feedback.

  7. overall internet service receives better feedback than mobile service
     suggesting the company should focus on improving its mobile network quality
     and customer support.
''')


dataset loaded successfully
  rows    : 12
  columns : 6
  columns : ['review_id', 'customer_name', 'service_type', 'branch_city', 'rating', 'review_text']

  review_id  customer_name    service_type    branch_city      rating  review_text
        901  Ali              Internet        Toronto               5  Excellent internet speed and very helpful customer support.
        902  Sara             Mobile          Ottawa                2  The mobile service was slow and the signal was weak.
        903  John             Internet        Vaughan               4  Good connection quality and easy setup process.
        904  Mina             Mobile          Markham               3  The service was okay but billing took too long.
        905  David            Internet        Toronto               1  Terrible experience, poor signal and rude support staff.

clean text preview
  original : Excellent internet speed and very helpful customer support.
  cleaned  : ['excellent', 'internet', 'speed